# 01 Data Exploration (Tier-Aware)

Purpose: profile and clean CORD-19 metadata for downstream RAG experiments.

Input: `1_data/raw/metadata.csv`
Output: `1_data/processed/papers_clean.parquet`, `1_data/processed/tier_manifest.json`

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
RAW = ROOT / '1_data' / 'raw' / 'metadata.csv'
OUT_DIR = ROOT / '1_data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIER_NAME = 'tier1'
TIER_SIZE = 1000
SEED = 42

assert RAW.exists(), f'Missing file: {RAW}'
df = pd.read_csv(RAW, low_memory=False)
df = df.drop_duplicates(subset=['cord_uid'])
df = df[df['abstract'].fillna('').str.len() > 100]
sample = df.sample(min(TIER_SIZE, len(df)), random_state=SEED).copy()

clean_path = OUT_DIR / 'papers_clean.parquet'
sample.to_parquet(clean_path, index=False)

manifest = {
    'tier_name': TIER_NAME,
    'target_size': TIER_SIZE,
    'actual_size': int(len(sample)),
    'seed': SEED,
    'source': str(RAW),
    'output': str(clean_path)
}
(OUT_DIR / 'tier_manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Wrote {len(sample):,} papers -> {clean_path}')
